# Reproducible GIWAXS processing: reciprocal-space images and radial profiles

This notebook is a clean, self-contained version of the processing workflow developed for the I07 Pilatus GIWAXS data. Starting from the raw `pilatus2-<scan>.hdf5` detector stack, matching `i07-<scan>.nxs` metadata, a `.poni` geometry calibration and an EDF detector mask, it produces:

## Acknowledgement and code provenance

This workflow is adapted from `i07_data_processing_clean.ipynb`, supplied by Dr Daniel Toolan. The accompanying `environment.yml` was also supplied by Dr Daniel Toolan and has subsequently been extended for this project, including the addition of `openpyxl` for Excel metadata export. These original materials provided the I07-oriented Python framework and software environment: the pyFAI/pyGIX imports, loading of the PONI detector geometry and EDF mask, Pilatus HDF5 frame access, reciprocal-space transformation and the basic frame-by-frame batch-processing pattern. This contribution is gratefully acknowledged.

The present notebook is a reorganised and extended implementation for this project. Project-specific additions include:

- a single documented settings cell and restart-and-run-all workflow;
- validation of detector, mask, calibration and metadata inputs;
- corrected placement of the measured quadrant at positive Qr without an additional image flip;
- explicit FULL/FR, IP and OOP sectors using `chi = arctan2(Qz, Qr)`;
- mean-intensity radial binning with empty detector bins retained as `NaN`;
- consistent scan-level colour scaling and black display of masked pixels;
- representative line-cut and paired IP/OOP plots;
- Q–time maps using the actual per-frame NeXus timing metadata rather than frame number or nominal exposure time;
- memory-safe scan batching, deterministic output folders, metadata tables and processing manifests;
- per-scan CSV/Excel frame–time–temperature tables and temperature trajectory plots.

The reorganisation and project-specific implementation were developed with assistance from OpenAI Codex.

The extraction implemented here does **not** perform background subtraction, incident-flux/monitor normalisation, peak fitting or coherence-length calculation. Those steps, if used, must be documented separately.

## Outputs

- calibrated reciprocal-space Qr–Qz images with a common colour scale within each scan;
- raw (not log-transformed) 1D mean-intensity CSV profiles for FULL/FR, IP and OOP sectors;
- five representative line-cut plots for each direction;
- five paired IP-versus-OOP plots;
- Q-versus-actual-elapsed-time intensity maps for FULL/FR, IP and OOP;
- per-frame timing/temperature metadata in CSV and Excel format;
- temperature-versus-time, temperature-versus-frame and dual-axis temperature plots;
- a JSON record of all processing settings.

## Run order

1. Change only the paths, scan list and documented settings in **Cell 3**.
2. Restart the kernel.
3. Use **Run All**.

The integrations are azimuthal means, not summed detector counts. Consequently the FULL/FR curve is not expected to equal IP + OOP or always exceed the directional curves.


In [ ]:
# Imports and software-version record

from pathlib import Path
import gc
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import h5py
import fabio
import pyFAI
import pygix
import openpyxl

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("h5py:", h5py.__version__)
print("FabIO:", getattr(fabio, "version", "unknown"))
print("pyFAI:", getattr(pyFAI, "version", "unknown"))
print("pygix:", getattr(pygix, "__version__", "unknown"))
print("openpyxl:", openpyxl.__version__)


In [ ]:
# USER SETTINGS — this is the only cell normally edited

DATA_DIR = Path(r"/Users/omar/Desktop/GIWAXS/GIWAXS Data/Data/All")
OUTPUT_DIR = Path(r"/Users/omar/Desktop/GIWAXS/GIWAXS Data/July_2025/Images/Reproducible_Processing")

PONI_FILE = Path(r"/Users/omar/Downloads/Toolan_Code_for_Mac/AgBH PONI.poni")
MASK_FILE = Path(r"/Users/omar/Downloads/Toolan_Code_for_Mac/AgBH Mask .edf")

# Add one or more scan numbers.
SCAN_NUMBERS = [587178, 587184, 587225]

# Experimental geometry and corrections used in the established workflow.
INCIDENT_ANGLE_DEG = 0.12
SAMPLE_ORIENTATION = 3
POLARIZATION_FACTOR = 1.0
CORRECT_SOLID_ANGLE = True
TRANSFORM_METHOD = "nearest"

# Reciprocal-space and 1D radial binning.
Q_MIN = 0.01
Q_MAX = 2.50
RADIAL_BINS = 400
MAP_BINS = 500

# chi is defined as arctan2(Qz, Qr): 0° is IP/horizontal; 90° is OOP/vertical.
SECTORS_DEG = {
    "FULL": (0.0, 90.0),   # full radial quadrant (FR)
    "IP":   (0.0, 20.0),
    "OOP":  (70.0, 90.0),
}
# These 20° directional sectors reproduce the current validated workflow.
# If the final analysis instead adopts 30° sectors, change both to IP 0–30°
# and OOP 60–90°, then regenerate every compared scan; do not mix definitions.

# Five equally spaced traces are used for the representative line plots.
REPRESENTATIVE_LINE_COUNT = 5
SKIP_FRAME_ZERO_FOR_REPRESENTATIVE_LINES = True

# Save one reciprocal-space image for every detector frame.
# Change to "representative" only when a storage-saving preview is wanted.
RECIPROCAL_IMAGE_MODE = "all"  # "all" or "representative"
RECIPROCAL_IMAGE_COUNT = 5
SAVE_RECIPROCAL_NUMERIC_NPZ = False  # optional transformed-map cache

CMAP_NAME = "turbo"
COLOUR_PERCENTILES = (1.0, 99.9)
FIGURE_DPI = 300

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Scans:", SCAN_NUMBERS)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)


## Calibration and mask initialisation

The AgBh-derived PONI file supplies detector distance, beam centre, rotations and wavelength. The corresponding EDF mask removes invalid pixels and detector-module gaps. The same geometry and mask are applied to every selected sample scan collected in this detector configuration.


In [ ]:
# Load the PONI geometry and detector mask into pyGIX

for required_file in (PONI_FILE, MASK_FILE):
    if not required_file.is_file():
        raise FileNotFoundError(f"Required calibration file not found: {required_file}")

AI = pyFAI.load(str(PONI_FILE))
MASK_DATA = np.asarray(fabio.open(str(MASK_FILE)).data)

PG = pygix.Transform()
PG.load(AI)
PG.maskfile = str(MASK_FILE)
PG.incident_angle = INCIDENT_ANGLE_DEG
PG.sample_orientation = SAMPLE_ORIENTATION

wavelength_angstrom = (
    float(AI.wavelength) * 1e10 if getattr(AI, "wavelength", None) else np.nan
)

print("PONI loaded:", PONI_FILE)
print("Mask loaded:", MASK_FILE)
print("Mask shape:", MASK_DATA.shape)
print(f"Sample–detector distance: {AI.dist:.6f} m")
print(f"Wavelength from PONI: {wavelength_angstrom:.6f} Å")
print(f"Incident angle: {INCIDENT_ANGLE_DEG:.3f}°")
print("Sample orientation:", SAMPLE_ORIENTATION)


## Input, timing and plotting helpers

Elapsed time is read from `entry/instrument/scanTimer/value`. If that field is unavailable, the notebook falls back to Diamond point-start timestamps. It never converts frames to time by merely renaming the axis or multiplying by exposure time.


In [ ]:
# Input and metadata helpers

DETECTOR_DATASET = "/entry/data/data"
SCAN_TIMER_DATASET = "entry/instrument/scanTimer/value"
POINT_TIME_DATASET = "entry/diamond_scan/point_start_times"
TEMPERATURE_DATASET = "entry/instrument/temp3/value"


def detector_file(scan_number):
    return DATA_DIR / f"pilatus2-{scan_number}.hdf5"


def sha256_file(filename, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(filename).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def file_record(filename, include_hash=False):
    filename = Path(filename)
    stat = filename.stat()
    record = {
        "path": str(filename),
        "size_bytes": int(stat.st_size),
        "modified_time_unix": float(stat.st_mtime),
    }
    if include_hash:
        record["sha256"] = sha256_file(filename)
    return record


def nexus_file(scan_number):
    candidates = [
        DATA_DIR / f"i07-{scan_number}.nxs",
        DATA_DIR / f"pilatus2-{scan_number}.nxs",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"No NeXus metadata file was found for scan {scan_number}. Checked: {candidates}"
    )


def read_detector_frame(detector_data, frame_number):
    if detector_data.ndim == 4:
        frame = detector_data[frame_number, 0, :, :]
    elif detector_data.ndim == 3:
        frame = detector_data[frame_number, :, :]
    else:
        raise ValueError(f"Unsupported detector dataset shape: {detector_data.shape}")
    return np.asarray(frame, dtype=np.float32)


def load_frame_metadata(scan_number, number_frames):
    metadata_path = nexus_file(scan_number)

    with h5py.File(metadata_path, "r") as nxs:
        if SCAN_TIMER_DATASET in nxs:
            elapsed_seconds = np.asarray(nxs[SCAN_TIMER_DATASET][:], dtype=float).ravel()
            timing_source = SCAN_TIMER_DATASET
        elif POINT_TIME_DATASET in nxs:
            # Diamond point timestamps are milliseconds since epoch.
            elapsed_seconds = (
                np.asarray(nxs[POINT_TIME_DATASET][:], dtype=float).ravel() / 1000.0
            )
            timing_source = POINT_TIME_DATASET
        else:
            raise KeyError(
                f"No per-frame timing dataset was found in {metadata_path}"
            )

        if TEMPERATURE_DATASET in nxs:
            temperature_c = np.asarray(nxs[TEMPERATURE_DATASET][:], dtype=float).ravel()
            temperature_source = TEMPERATURE_DATASET
        else:
            temperature_c = np.full(number_frames, np.nan)
            temperature_source = None

    timing_values_in_file = int(elapsed_seconds.size)
    temperature_values_in_file = (
        int(temperature_c.size) if temperature_source is not None else 0
    )

    if elapsed_seconds.size < number_frames:
        raise ValueError(
            f"Scan {scan_number}: {number_frames} frames but only "
            f"{elapsed_seconds.size} timing values"
        )

    elapsed_seconds = elapsed_seconds[:number_frames]
    elapsed_seconds = elapsed_seconds - elapsed_seconds[0]

    if not np.all(np.isfinite(elapsed_seconds)):
        raise ValueError(f"Scan {scan_number}: timing contains non-finite values")
    if number_frames > 1 and np.any(np.diff(elapsed_seconds) <= 0):
        raise ValueError(f"Scan {scan_number}: timing is not strictly increasing")

    if temperature_c.size < number_frames:
        temperature_c = np.pad(
            temperature_c,
            (0, number_frames - temperature_c.size),
            constant_values=np.nan,
        )
    temperature_c = temperature_c[:number_frames]
    temperature_c[~np.isfinite(temperature_c)] = np.nan

    metadata = pd.DataFrame(
        {
            "frame": np.arange(number_frames, dtype=int),
            "elapsed_time_s": elapsed_seconds,
            "temperature_C": temperature_c,
        }
    )

    finite_temperature_count = int(np.isfinite(temperature_c).sum())
    if temperature_source is None or finite_temperature_count == 0:
        temperature_status = "missing"
    elif finite_temperature_count == number_frames:
        temperature_status = "complete"
    else:
        temperature_status = "partial"

    metadata.attrs.update(
        {
            "timing_source": timing_source,
            "timing_values_in_file": timing_values_in_file,
            "temperature_source": temperature_source,
            "temperature_status": temperature_status,
            "temperature_values_in_file": temperature_values_in_file,
            "temperature_finite_count": finite_temperature_count,
            "temperature_alignment": (
                "index matched to detector frame; extra values truncated; "
                "shortages padded with NaN"
            ),
        }
    )

    return metadata, metadata_path, timing_source


def representative_indices(number_frames, count=5, skip_zero=False):
    if number_frames <= 0:
        return np.array([], dtype=int)

    first = 1 if skip_zero and number_frames > 1 else 0
    available = number_frames - first
    count = min(int(count), available)

    if count <= 1:
        return np.array([first], dtype=int)

    return np.unique(
        np.rint(np.linspace(first, number_frames - 1, count)).astype(int)
    )


def centres_to_edges(values, start_at_zero=False):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        raise ValueError("Cannot make coordinate edges from an empty array")
    if values.size == 1:
        edges = np.array([values[0], values[0] + 1.0], dtype=float)
    else:
        midpoints = 0.5 * (values[:-1] + values[1:])
        first = values[0] - 0.5 * (values[1] - values[0])
        last = values[-1] + 0.5 * (values[-1] - values[-2])
        edges = np.concatenate(([first], midpoints, [last]))
    if start_at_zero:
        edges[0] = 0.0
    return edges


def positive_log10(values):
    values = np.asarray(values, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.log10(np.where(values > 0, values, np.nan))


def colour_limits(log_values, percentiles=COLOUR_PERCENTILES):
    values = np.asarray(log_values, dtype=float)
    valid = values[np.isfinite(values) & (values > -9)]
    if valid.size == 0:
        raise ValueError("No valid positive intensity values are available")
    return tuple(np.nanpercentile(valid, percentiles))


## Reciprocal-space transformation and radial sectors

The transformed quadrant is converted from negative Qxy to positive Qr and sorted so Qr increases left-to-right and Qz increases bottom-to-top. For each radial-Q bin, intensity is the mean of valid transformed pixels within the requested chi range.


In [ ]:
# Qr–Qz transformation and FULL/IP/OOP radial integration

Q_EDGES = np.linspace(Q_MIN, Q_MAX, RADIAL_BINS + 1)
Q_CENTRES = 0.5 * (Q_EDGES[:-1] + Q_EDGES[1:])


def reciprocal_map(detector_image):
    intensity_map, qxy, qz = PG.transform_reciprocal(
        detector_image,
        npt=(MAP_BINS, MAP_BINS),
        ip_range=(-Q_MAX, 0.05),
        op_range=(0.0, Q_MAX),
        polarization_factor=POLARIZATION_FACTOR,
        correctSolidAngle=CORRECT_SOLID_ANGLE,
        mask=MASK_DATA,
        method=TRANSFORM_METHOD,
        unit="A",
    )

    intensity_map = np.asarray(intensity_map, dtype=float)
    qxy = np.asarray(qxy, dtype=float)
    qz = np.asarray(qz, dtype=float)

    # The selected detector orientation returns the measured quadrant at negative Qxy.
    qr = -qxy
    qr_order = np.argsort(qr)
    qz_order = np.argsort(qz)

    qr = qr[qr_order]
    qz = qz[qz_order]
    intensity_map = intensity_map[qz_order, :][:, qr_order]

    intensity_map[~np.isfinite(intensity_map)] = np.nan
    intensity_map[intensity_map <= 1e-8] = np.nan

    return intensity_map, qr, qz


def radial_profiles_from_map(intensity_map, qr, qz):
    QR, QZ = np.meshgrid(qr, qz)
    radial_q = np.sqrt(QR**2 + QZ**2)
    chi_deg = np.degrees(np.arctan2(QZ, QR))

    base_mask = (
        np.isfinite(intensity_map)
        & np.isfinite(radial_q)
        & np.isfinite(chi_deg)
        & (QR >= 0)
        & (QZ >= 0)
        & (radial_q >= Q_MIN)
        & (radial_q <= Q_MAX)
    )

    profiles = {}
    valid_pixel_counts = {}

    for direction, (chi_min, chi_max) in SECTORS_DEG.items():
        sector_mask = base_mask & (chi_deg >= chi_min) & (chi_deg <= chi_max)
        q_selected = radial_q[sector_mask]
        intensity_selected = intensity_map[sector_mask]

        intensity_sum, _ = np.histogram(
            q_selected, bins=Q_EDGES, weights=intensity_selected
        )
        pixel_count, _ = np.histogram(q_selected, bins=Q_EDGES)

        profile = np.full(RADIAL_BINS, np.nan, dtype=float)
        valid = pixel_count > 0
        profile[valid] = intensity_sum[valid] / pixel_count[valid]
        profile[profile <= 1e-8] = np.nan
        profiles[direction] = profile
        valid_pixel_counts[direction] = pixel_count.astype(int)

    return profiles, valid_pixel_counts


## Output plotting helpers

All reciprocal-space images from one scan share a colour scale. The three time maps from one scan also share a colour scale. Logarithms are used only for visualisation; saved profile CSV values remain on the original linear intensity scale.


In [ ]:
# Plotting and data-output helpers

DIRECTION_TITLES = {
    "FULL": "FULL/FR 0–90°",
    "IP": "IP 0–20°",
    "OOP": "OOP 70–90°",
}

# Give masked/invalid reciprocal-space pixels an opaque black colour.
PLOT_CMAP = matplotlib.colormaps[CMAP_NAME].copy()
PLOT_CMAP.set_bad(color="black", alpha=1.0)


def save_frame_time_temperature_outputs(
    metadata, metadata_path, timing_source, scan_number, output_folder
):
    output_folder.mkdir(parents=True, exist_ok=True)

    export_table = metadata.rename(columns={"frame": "frame_index_0based"})
    stem = f"pilatus2-{scan_number}_frame_time_temperature"
    csv_file = output_folder / f"{stem}.csv"
    excel_file = output_folder / f"{stem}.xlsx"
    export_table.to_csv(csv_file, index=False)

    elapsed_seconds = metadata["elapsed_time_s"].to_numpy(dtype=float)
    frame_indices = metadata["frame"].to_numpy(dtype=int)
    temperature_c = metadata["temperature_C"].to_numpy(dtype=float)
    finite_temperature = np.isfinite(temperature_c)
    finite_count = int(finite_temperature.sum())
    temperature_status = metadata.attrs.get("temperature_status", "missing")

    if finite_count:
        finite_values = temperature_c[finite_temperature]
        temperature_min = float(np.min(finite_values))
        temperature_max = float(np.max(finite_values))
        temperature_first = float(finite_values[0])
        temperature_last = float(finite_values[-1])
    else:
        temperature_min = None
        temperature_max = None
        temperature_first = None
        temperature_last = None

    provenance = pd.DataFrame(
        [
            ("scan_number", int(scan_number)),
            ("nexus_file", str(metadata_path)),
            ("frame_numbering", "zero-based detector-frame index"),
            ("timing_source", timing_source),
            ("timing_values_in_file", metadata.attrs.get("timing_values_in_file")),
            ("elapsed_time_zero", "first detector frame"),
            ("temperature_source", metadata.attrs.get("temperature_source")),
            ("temperature_status", temperature_status),
            ("temperature_values_in_file", metadata.attrs.get("temperature_values_in_file")),
            ("temperature_finite_count", finite_count),
            ("temperature_alignment", metadata.attrs.get("temperature_alignment")),
            ("temperature_interpolation", False),
            ("temperature_smoothing", False),
        ],
        columns=["field", "value"],
    )

    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        export_table.to_excel(
            writer, sheet_name="frame_metadata", index=False, freeze_panes=(1, 0)
        )
        provenance.to_excel(
            writer, sheet_name="provenance", index=False, freeze_panes=(1, 0)
        )
        metadata_sheet = writer.sheets["frame_metadata"]
        metadata_sheet.column_dimensions["A"].width = 23
        metadata_sheet.column_dimensions["B"].width = 20
        metadata_sheet.column_dimensions["C"].width = 18
        provenance_sheet = writer.sheets["provenance"]
        provenance_sheet.column_dimensions["A"].width = 32
        provenance_sheet.column_dimensions["B"].width = 80

    plot_files = []
    if finite_count == 0:
        print(
            f"  Scan {scan_number}: no finite temperature data; "
            "saved the CSV/Excel table and skipped temperature plots."
        )
    else:
        if temperature_status == "partial":
            print(
                f"  Scan {scan_number}: temperature data are partial; "
                "plots retain gaps at missing readings."
            )

        plot_specs = [
            (
                elapsed_seconds,
                "Elapsed time [s]",
                "temperature vs elapsed time",
                output_folder / f"pilatus2-{scan_number}_temperature_vs_elapsed_time.png",
            ),
            (
                frame_indices,
                "Frame index (0-based)",
                "temperature vs frame",
                output_folder / f"pilatus2-{scan_number}_temperature_vs_frame.png",
            ),
        ]

        for x_values, x_label, title_suffix, output_file in plot_specs:
            fig, ax = plt.subplots(
                figsize=(7.2, 4.8), dpi=150, constrained_layout=True
            )
            ax.plot(
                x_values, temperature_c, color="tab:red", linewidth=1.4,
                marker="o", markersize=2.5
            )
            ax.set_xlabel(x_label)
            ax.set_ylabel("Temperature [°C]")
            ax.set_title(f"pilatus2-{scan_number}: {title_suffix}")
            ax.grid(alpha=0.2)
            fig.savefig(output_file, dpi=FIGURE_DPI, bbox_inches="tight")
            plt.close(fig)
            plot_files.append(output_file)

        dual_file = output_folder / (
            f"pilatus2-{scan_number}_temperature_vs_elapsed_time_with_frame_axis.png"
        )
        fig, ax = plt.subplots(
            figsize=(7.2, 4.8), dpi=150, constrained_layout=True
        )
        ax.plot(
            elapsed_seconds, temperature_c, color="tab:red", linewidth=1.4,
            marker="o", markersize=2.5
        )
        ax.set_xlabel("Elapsed time [s]")
        ax.set_ylabel("Temperature [°C]")
        ax.set_title(
            f"pilatus2-{scan_number}: temperature vs elapsed time and frame"
        )
        ax.grid(alpha=0.2)

        if elapsed_seconds.size == 1 or elapsed_seconds[-1] == 0:
            ax.set_xlim(-0.5, 0.5)
        else:
            ax.set_xlim(0.0, float(elapsed_seconds[-1]))

        top_axis = ax.twiny()
        top_axis.set_xlim(ax.get_xlim())
        tick_rows = representative_indices(
            len(metadata), count=min(7, len(metadata)), skip_zero=False
        )
        top_axis.set_xticks(elapsed_seconds[tick_rows])
        top_axis.set_xticklabels(frame_indices[tick_rows].astype(str))
        top_axis.set_xlabel("Frame index (0-based)")
        fig.savefig(dual_file, dpi=FIGURE_DPI, bbox_inches="tight")
        plt.close(fig)
        plot_files.append(dual_file)

    return {
        "csv_file": str(csv_file),
        "excel_file": str(excel_file),
        "plot_files": [str(path) for path in plot_files],
        "temperature_status": temperature_status,
        "temperature_finite_count": finite_count,
        "temperature_min_C": temperature_min,
        "temperature_max_C": temperature_max,
        "temperature_first_C": temperature_first,
        "temperature_last_C": temperature_last,
    }


def save_reciprocal_image(
    intensity_map, qr, qz, scan_number, frame_number, elapsed_seconds,
    output_folder, vmin, vmax
):
    log_map = np.ma.masked_invalid(positive_log10(intensity_map))

    fig, ax = plt.subplots(figsize=(7.4, 6.2), dpi=150, constrained_layout=True)
    ax.set_facecolor("black")
    image = ax.imshow(
        log_map,
        origin="lower",
        aspect="equal",
        extent=[qr.min(), qr.max(), qz.min(), qz.max()],
        interpolation="nearest",
        cmap=PLOT_CMAP,
        vmin=vmin,
        vmax=vmax,
    )
    colourbar = fig.colorbar(image, ax=ax, pad=0.03)
    colourbar.set_label("log10 intensity [a.u.]")

    ax.set_xlim(0, Q_MAX)
    ax.set_ylim(0, Q_MAX)
    ax.set_xlabel("Qr / Qxy [Å$^{-1}$]")
    ax.set_ylabel("Qz [Å$^{-1}$]")
    ax.set_title(
        f"pilatus2-{scan_number}: reciprocal-space frame {frame_number}, "
        f"t = {elapsed_seconds:.2f} s"
    )

    output_file = output_folder / f"frame_{frame_number:04d}_Qr_Qz.png"
    fig.savefig(output_file, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

    if SAVE_RECIPROCAL_NUMERIC_NPZ:
        np.savez_compressed(
            output_folder / f"frame_{frame_number:04d}_Qr_Qz_numeric.npz",
            intensity=np.asarray(intensity_map, dtype=np.float32),
            qr_A_1=np.asarray(qr, dtype=np.float32),
            qz_A_1=np.asarray(qz, dtype=np.float32),
        )

    return output_file


def profile_dataframe(profile_matrix):
    dataframe = pd.DataFrame({"Q_A-1": Q_CENTRES})
    for frame_number, profile in enumerate(profile_matrix):
        dataframe[str(frame_number)] = profile
    return dataframe


def save_representative_line_plot(
    dataframe, direction, metadata, selected_frames, output_folder, scan_number
):
    q = dataframe["Q_A-1"].to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=150, constrained_layout=True)

    for frame_number in selected_frames:
        log_intensity = positive_log10(
            dataframe[str(frame_number)].to_numpy(dtype=float)
        )
        elapsed = metadata.loc[frame_number, "elapsed_time_s"]
        ax.plot(
            q,
            log_intensity,
            linewidth=1.4,
            label=f"t = {elapsed:.1f} s (frame {frame_number})",
        )

    ax.set_xlim(Q_MIN, Q_MAX)
    ax.set_xlabel("Q [Å$^{-1}$]")
    ax.set_ylabel("log10 mean intensity [a.u.]")
    ax.set_title(f"pilatus2-{scan_number}: {DIRECTION_TITLES[direction]} radial profiles")
    ax.legend(frameon=False, fontsize=8)

    output_file = output_folder / f"pilatus2-{scan_number}_{direction}_five_time_linecuts.png"
    fig.savefig(output_file, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)
    return output_file


def save_ip_oop_pair(
    dataframes, metadata, frame_number, output_folder, scan_number
):
    q = dataframes["IP"]["Q_A-1"].to_numpy(dtype=float)
    ip_log = positive_log10(dataframes["IP"][str(frame_number)].to_numpy(dtype=float))
    oop_log = positive_log10(dataframes["OOP"][str(frame_number)].to_numpy(dtype=float))
    elapsed = metadata.loc[frame_number, "elapsed_time_s"]

    fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=150, constrained_layout=True)
    ax.plot(q, ip_log, linewidth=1.5, label="IP 0–20°")
    ax.plot(q, oop_log, linewidth=1.5, linestyle="--", label="OOP 70–90°")
    ax.set_xlim(Q_MIN, Q_MAX)
    ax.set_xlabel("Q [Å$^{-1}$]")
    ax.set_ylabel("log10 mean intensity [a.u.]")
    ax.set_title(
        f"pilatus2-{scan_number}: IP vs OOP at t = {elapsed:.1f} s "
        f"(frame {frame_number})"
    )
    ax.legend(frameon=False)

    output_file = output_folder / (
        f"pilatus2-{scan_number}_IP_vs_OOP_time_{elapsed:08.2f}s_frame_{frame_number:04d}.png"
    )
    fig.savefig(output_file, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)
    return output_file


def save_time_map(
    dataframe, direction, elapsed_seconds, output_folder, scan_number, vmin, vmax
):
    q = dataframe["Q_A-1"].to_numpy(dtype=float)
    intensity = dataframe.drop(columns="Q_A-1").to_numpy(dtype=float).T
    log_intensity = positive_log10(intensity)

    time_edges = centres_to_edges(elapsed_seconds, start_at_zero=True)
    q_edges = centres_to_edges(q)

    fig, ax = plt.subplots(figsize=(8.2, 6.2), dpi=150, constrained_layout=True)
    image = ax.pcolormesh(
        time_edges,
        q_edges,
        log_intensity.T,
        shading="auto",
        cmap=PLOT_CMAP,
        vmin=vmin,
        vmax=vmax,
    )
    colourbar = fig.colorbar(image, ax=ax, pad=0.03)
    colourbar.set_label("log10 mean intensity [a.u.]")

    ax.set_xlim(0, time_edges[-1])
    ax.set_ylim(Q_MIN, Q_MAX)
    ax.set_xlabel("Elapsed time [s]")
    ax.set_ylabel("Q [Å$^{-1}$]")
    ax.set_title(
        f"pilatus2-{scan_number}: {DIRECTION_TITLES[direction]} intensity vs elapsed time"
    )

    output_file = output_folder / f"pilatus2-{scan_number}_{direction}_1D_map_vs_time.png"
    fig.savefig(output_file, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)
    return output_file


## Scan-processing driver

Each detector frame is loaded, transformed and released before the next frame is processed. Only the compact 1D profiles are retained in memory. This avoids accumulating full detector or reciprocal-space stacks.


In [ ]:
# Validate and process one scan


def validate_scan(scan_number):
    raw_path = detector_file(scan_number)
    metadata_path = nexus_file(scan_number)

    if not raw_path.is_file():
        raise FileNotFoundError(f"Detector file not found: {raw_path}")

    with h5py.File(raw_path, "r") as raw_file:
        if DETECTOR_DATASET not in raw_file:
            raise KeyError(f"{DETECTOR_DATASET} not found in {raw_path}")
        detector_data = raw_file[DETECTOR_DATASET]
        number_frames = detector_data.shape[0]
        detector_shape = detector_data.shape[-2:]

    if tuple(detector_shape) != tuple(MASK_DATA.shape):
        raise ValueError(
            f"Scan {scan_number}: detector frame shape {detector_shape} does not "
            f"match mask shape {MASK_DATA.shape}"
        )

    metadata, _, timing_source = load_frame_metadata(scan_number, number_frames)

    print(
        f"Scan {scan_number}: {number_frames} frames, detector {detector_shape}, "
        f"elapsed 0–{metadata['elapsed_time_s'].iloc[-1]:.2f} s"
    )
    print("  raw:", raw_path)
    print("  metadata:", metadata_path)
    print("  timing source:", timing_source)

    return number_frames


def process_scan(scan_number):
    raw_path = detector_file(scan_number)
    metadata_path = nexus_file(scan_number)
    number_frames = validate_scan(scan_number)
    metadata, _, timing_source = load_frame_metadata(scan_number, number_frames)

    scan_folder = OUTPUT_DIR / f"scan_{scan_number}"
    reciprocal_folder = scan_folder / "Reciprocal_Space"
    profile_folder = scan_folder / "Individual_1D"
    lineplot_folder = profile_folder / "Line_Plots"
    time_map_folder = profile_folder / "Time_Maps"
    metadata_folder = scan_folder / "Metadata"

    for folder in (
        scan_folder, reciprocal_folder, profile_folder, lineplot_folder,
        time_map_folder, metadata_folder
    ):
        folder.mkdir(parents=True, exist_ok=True)

    # Save lightweight frame/time/temperature outputs before the long image loop.
    legacy_metadata_file = scan_folder / f"pilatus2-{scan_number}_frame_metadata.csv"
    metadata.to_csv(legacy_metadata_file, index=False)
    metadata_outputs = save_frame_time_temperature_outputs(
        metadata, metadata_path, timing_source, scan_number, metadata_folder
    )

    if RECIPROCAL_IMAGE_MODE == "all":
        reciprocal_frames = np.arange(number_frames, dtype=int)
    elif RECIPROCAL_IMAGE_MODE == "representative":
        reciprocal_frames = representative_indices(
            number_frames, RECIPROCAL_IMAGE_COUNT, skip_zero=False
        )
    else:
        raise ValueError('RECIPROCAL_IMAGE_MODE must be "representative" or "all"')

    scale_frames = representative_indices(
        number_frames, max(RECIPROCAL_IMAGE_COUNT, 5), skip_zero=False
    )

    # First, obtain a stable reciprocal-space colour scale from representative frames.
    scale_values = []
    with h5py.File(raw_path, "r") as raw_file:
        detector_data = raw_file[DETECTOR_DATASET]
        for frame_number in scale_frames:
            image = read_detector_frame(detector_data, int(frame_number))
            intensity_map, _, _ = reciprocal_map(image)
            values = positive_log10(intensity_map)
            valid = values[np.isfinite(values) & (values > -9)]
            if valid.size:
                scale_values.append(valid)
            del image, intensity_map, values
            gc.collect()

    if not scale_values:
        raise ValueError(f"Scan {scan_number}: no valid reciprocal-space intensity values")

    reciprocal_vmin, reciprocal_vmax = colour_limits(np.concatenate(scale_values))
    reciprocal_frame_set = set(int(value) for value in reciprocal_frames)

    profile_matrices = {
        direction: np.full((number_frames, RADIAL_BINS), np.nan, dtype=float)
        for direction in SECTORS_DEG
    }
    pixel_count_matrices = {
        direction: np.zeros((number_frames, RADIAL_BINS), dtype=np.int32)
        for direction in SECTORS_DEG
    }

    with h5py.File(raw_path, "r") as raw_file:
        detector_data = raw_file[DETECTOR_DATASET]

        for frame_number in range(number_frames):
            print(
                f"\rScan {scan_number}: processing frame {frame_number + 1}/{number_frames}",
                end="",
                flush=True,
            )

            image = read_detector_frame(detector_data, frame_number)
            intensity_map, qr, qz = reciprocal_map(image)
            profiles, valid_pixel_counts = radial_profiles_from_map(
                intensity_map, qr, qz
            )

            for direction in SECTORS_DEG:
                profile_matrices[direction][frame_number, :] = profiles[direction]
                pixel_count_matrices[direction][frame_number, :] = (
                    valid_pixel_counts[direction]
                )

            if frame_number in reciprocal_frame_set:
                save_reciprocal_image(
                    intensity_map,
                    qr,
                    qz,
                    scan_number,
                    frame_number,
                    metadata.loc[frame_number, "elapsed_time_s"],
                    reciprocal_folder,
                    reciprocal_vmin,
                    reciprocal_vmax,
                )

            del image, intensity_map, qr, qz, profiles, valid_pixel_counts
            if frame_number % 10 == 0:
                gc.collect()

    print("")

    dataframes = {}
    for direction, matrix in profile_matrices.items():
        dataframe = profile_dataframe(matrix)
        dataframes[direction] = dataframe
        csv_file = profile_folder / f"pilatus2-{scan_number}_{direction}_1Dintegrations.csv"
        dataframe.to_csv(csv_file, index=False)

        count_dataframe = profile_dataframe(pixel_count_matrices[direction])
        count_file = profile_folder / (
            f"pilatus2-{scan_number}_{direction}_valid_pixel_counts.csv"
        )
        count_dataframe.to_csv(count_file, index=False)

    selected_frames = representative_indices(
        number_frames,
        REPRESENTATIVE_LINE_COUNT,
        skip_zero=SKIP_FRAME_ZERO_FOR_REPRESENTATIVE_LINES,
    )

    for direction, dataframe in dataframes.items():
        save_representative_line_plot(
            dataframe,
            direction,
            metadata,
            selected_frames,
            lineplot_folder,
            scan_number,
        )

    for frame_number in selected_frames:
        save_ip_oop_pair(
            dataframes,
            metadata,
            int(frame_number),
            lineplot_folder,
            scan_number,
        )

    # One shared colour scale makes FULL/IP/OOP maps visually comparable within a scan.
    time_map_values = []
    for dataframe in dataframes.values():
        values = positive_log10(dataframe.drop(columns="Q_A-1").to_numpy(dtype=float))
        valid = values[np.isfinite(values) & (values > -9)]
        if valid.size:
            time_map_values.append(valid)

    shared_vmin, shared_vmax = colour_limits(np.concatenate(time_map_values))
    elapsed_seconds = metadata["elapsed_time_s"].to_numpy(dtype=float)

    for direction, dataframe in dataframes.items():
        save_time_map(
            dataframe,
            direction,
            elapsed_seconds,
            time_map_folder,
            scan_number,
            shared_vmin,
            shared_vmax,
        )

    software_versions = {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
        "h5py": h5py.__version__,
        "pyFAI": str(getattr(pyFAI, "version", "unknown")),
        "pygix": str(getattr(pygix, "__version__", "unknown")),
        "openpyxl": openpyxl.__version__,
    }

    parameters = {
        "scan_number": int(scan_number),
        "raw_file": file_record(raw_path, include_hash=False),
        "nexus_file": file_record(metadata_path, include_hash=False),
        "timing_source": timing_source,
        "elapsed_time_zeroed_to_first_frame": True,
        "elapsed_time_total_s": float(elapsed_seconds[-1]),
        "frame_numbering": "zero_based",
        "temperature_source": metadata.attrs.get("temperature_source"),
        "temperature_status": metadata_outputs["temperature_status"],
        "temperature_alignment": metadata.attrs.get("temperature_alignment"),
        "temperature_interpolation": False,
        "temperature_smoothing": False,
        "temperature_finite_count": metadata_outputs["temperature_finite_count"],
        "temperature_min_C": metadata_outputs["temperature_min_C"],
        "temperature_max_C": metadata_outputs["temperature_max_C"],
        "temperature_first_C": metadata_outputs["temperature_first_C"],
        "temperature_last_C": metadata_outputs["temperature_last_C"],
        "frame_metadata_csv": metadata_outputs["csv_file"],
        "frame_time_temperature_xlsx": metadata_outputs["excel_file"],
        "temperature_plot_files": metadata_outputs["plot_files"],
        "temperature_plots_generated": bool(metadata_outputs["plot_files"]),
        "temperature_dual_axis_method": (
            "top frame ticks positioned at their actual elapsed times"
        ),
        "poni_file": file_record(PONI_FILE, include_hash=True),
        "mask_file": file_record(MASK_FILE, include_hash=True),
        "detector_model": str(AI.detector),
        "sample_detector_distance_m": float(AI.dist),
        "wavelength_A": float(wavelength_angstrom),
        "incident_angle_deg": INCIDENT_ANGLE_DEG,
        "sample_orientation": SAMPLE_ORIENTATION,
        "polarization_factor": POLARIZATION_FACTOR,
        "correct_solid_angle": CORRECT_SOLID_ANGLE,
        "transform_method": TRANSFORM_METHOD,
        "q_min_A-1": Q_MIN,
        "q_max_A-1": Q_MAX,
        "radial_bins": RADIAL_BINS,
        "map_bins": MAP_BINS,
        "sectors_deg": SECTORS_DEG,
        "intensity_statistic": "azimuthal mean of valid transformed pixels per radial-Q bin",
        "monitor_or_exposure_normalization": False,
        "background_subtraction": False,
        "reciprocal_image_mode": RECIPROCAL_IMAGE_MODE,
        "reciprocal_numeric_npz": SAVE_RECIPROCAL_NUMERIC_NPZ,
        "reciprocal_image_frames": reciprocal_frames.tolist(),
        "representative_line_frames": selected_frames.tolist(),
        "colour_percentiles": COLOUR_PERCENTILES,
        "software_versions": software_versions,
    }

    parameters_file = scan_folder / f"pilatus2-{scan_number}_processing_parameters.json"
    with parameters_file.open("w", encoding="utf-8") as handle:
        json.dump(parameters, handle, indent=2)

    print(f"Completed scan {scan_number}")
    print("Output:", scan_folder)

    return {
        "scan_number": int(scan_number),
        "frames": int(number_frames),
        "elapsed_seconds": float(elapsed_seconds[-1]),
        "metadata_excel": metadata_outputs["excel_file"],
        "output_folder": str(scan_folder),
    }


## Input audit

Run this lightweight check before the long processing cell. It confirms that every detector file, NeXus timing file and detector/mask shape is compatible.


In [ ]:
# Audit all requested inputs before processing

for scan_number in SCAN_NUMBERS:
    validate_scan(scan_number)

print("All selected scans passed the input audit.")


## Run the complete workflow

This is the only computationally long cell. Existing files with the same names are replaced, allowing a scan to be rerun after changing documented settings.


In [ ]:
# Process every selected scan

processing_results = []

for scan_number in SCAN_NUMBERS:
    processing_results.append(process_scan(scan_number))

results_table = pd.DataFrame(processing_results)
results_table


In [ ]:
# Export the raw FULL/FR Q-versus-time map arrays for external plotting
# This cell does not recalculate or modify any existing analysis.

MAP_EXPORTS = {
    587207: "map_left.csv",
    587217: "map_right.csv",
}

MAP_EXPORT_DIR = OUTPUT_DIR / "Map_CSV_Exports"
MAP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for export_scan, export_name in MAP_EXPORTS.items():
    scan_folder = OUTPUT_DIR / f"scan_{export_scan}"
    profile_path = (
        scan_folder / "Individual_1D"
        / f"pilatus2-{export_scan}_FULL_1Dintegrations.csv"
    )
    metadata_path = scan_folder / f"pilatus2-{export_scan}_frame_metadata.csv"

    for required_path in (profile_path, metadata_path):
        if not required_path.is_file():
            raise FileNotFoundError(
                f"Required processed file not found: {required_path}. "
                f"Process scan {export_scan} before running this export cell."
            )

    profile_table = pd.read_csv(profile_path)
    frame_metadata = pd.read_csv(metadata_path)

    q_values = profile_table["Q_A-1"].to_numpy(dtype=float)
    frame_indices = frame_metadata["frame"].to_numpy(dtype=int)
    elapsed_seconds = frame_metadata["elapsed_time_s"].to_numpy(dtype=float)
    frame_columns = [str(frame) for frame in frame_indices]

    if frame_columns != list(profile_table.columns[1:]):
        raise ValueError(
            f"Scan {export_scan}: profile columns do not match metadata frame order"
        )
    if elapsed_seconds.size > 1 and np.any(np.diff(elapsed_seconds) <= 0):
        raise ValueError(f"Scan {export_scan}: elapsed times are not strictly increasing")

    # save_time_map() uses this (time, Q) orientation before plotting.
    map_time_q = profile_table[frame_columns].to_numpy(dtype=float).T

    print(f"\nScan {export_scan}")
    print("Underlying map orientation: (time, Q)")
    print("Underlying array shape (time, Q):", map_time_q.shape)
    print(f"Q min/max: {q_values.min():.7g}, {q_values.max():.7g} Å^-1")
    print(
        f"Elapsed-time min/max: {elapsed_seconds.min():.12g}, "
        f"{elapsed_seconds.max():.12g} s"
    )
    print("Intensity values: raw mean intensity; log10 has not been applied")

    # Required output orientation: one Q row and one elapsed-time column per frame.
    map_q_time = map_time_q.T
    time_headers = [f"{value:.12g}" for value in elapsed_seconds]
    if len(time_headers) != len(set(time_headers)):
        raise ValueError(
            f"Scan {export_scan}: formatted elapsed-time headers are not unique"
        )

    export_table = pd.DataFrame(map_q_time, columns=time_headers)
    export_table.insert(0, "Q", q_values)
    output_path = MAP_EXPORT_DIR / export_name
    export_table.to_csv(output_path, index=False)
    print("Export array shape (Q, time):", map_q_time.shape)
    print("Saved:", output_path)


## Interpretation and reporting notes

- **FULL/FR:** chi = 0–90°, the entire available transformed quadrant.
- **IP:** chi = 0–20°, close to the substrate plane.
- **OOP:** chi = 70–90°, close to the surface normal.
- chi is calculated as `arctan2(Qz, Qr)` after placing the measured quadrant at positive Qr and Qz.
- Each profile is the **mean intensity per valid transformed pixel** in a radial-Q bin. FULL is therefore an angular average, not IP + OOP.
- CSV intensities are linear. `log10` is applied only to figures.
- Time-map x coordinates come from actual per-frame scan timing metadata and include acquisition overhead.
- The processing JSON written for every scan is the authoritative record of parameters and software versions for a dissertation methods/audit trail.
